# ⚽ Manchester United Analytics Platform
## Complete Data Pipeline: Transform & Load to Snowflake

**Pipeline:** DBFS → PySpark Transformation → Snowflake DW  
**Data:** 900+ matches, 50M+ events  
**Stack:** Databricks + Snowflake

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ⚠️ IMPORTANT: Set correct data path
DATA_PATH = "dbfs:/statsbomb-data"  # Your actual path!

print("⚽ Manchester United Analytics Platform")
print("="*60)
print(f"🔥 Spark Version: {spark.version}")
print(f"📂 Data Path: {DATA_PATH}")
print("="*60)

⚽ Manchester United Analytics Platform
🔥 Spark Version: 3.4.1
📂 Data Path: dbfs:/statsbomb-data


## 🔐 Snowflake Configuration

In [0]:
SNOWFLAKE_OPTIONS = {
    "sfUrl": "ZQCYYEO-VG84200.snowflakecomputing.com",   
    "sfUser": "ABDALLAHMANNTD",           
    "sfPassword": "\"2]VaS4Cav]X8CR",                
    "sfDatabase": "FOOTBALL_ANALYTICS",
    "sfSchema": "MANUTD",
    "sfWarehouse": "ANALYTICS_WH"
}
 

In [0]:
# Test connection
print("🔍 Testing Snowflake connection...")

try:
    # Create simple test DataFrame
    test_df = spark.createDataFrame([(1, "test")], ["id", "name"])
    
    # Try to write
    test_df.write \
        .format("snowflake") \
        .options(**SNOWFLAKE_OPTIONS) \
        .option("dbtable", "CONNECTION_TEST") \
        .mode("overwrite") \
        .save()
    
    print("✅ Connection successful!")
    
    # Clean up test table
    # (You can manually delete it in Snowflake later)
    
except Exception as e:
    print(f"❌ Connection failed!")
    print(f"Error: {str(e)}")
    print("\n💡 Troubleshooting:")
    print("1. Check your account URL format")
    print("2. Verify username/password")
    print("3. Make sure database, schema, and warehouse exist in Snowflake")
    print("4. Check if warehouse is running (can be suspended)")
    print("\n📝 Run this in Snowflake to get your account info:")
    print("   SELECT CURRENT_ACCOUNT(), CURRENT_REGION();")
    raise


🔍 Testing Snowflake connection...
✅ Connection successful!


## 📊 1. Load Competitions


In [0]:
# Verify data exists
print("🔍 Verifying data location...")
try:
    files = dbutils.fs.ls(DATA_PATH)
    print(f"✅ Found {len(files)} items:")
    for f in files:
        print(f"   📁 {f.name}")
except Exception as e:
    print(f"❌ Error accessing {DATA_PATH}")
    print(f"   {e}")
    print("\n💡 Try checking the path in Data → Browse DBFS")

🔍 Verifying data location...
✅ Found 5 items:
   📁 competitions.json
   📁 events/
   📁 lineups/
   📁 matches/
   📁 three-sixty/


In [0]:
# Read competitions 
competitions_path = f"{DATA_PATH}/competitions.json"
print(f"📥 Reading: {competitions_path}")

competitions_df = (
    spark.read
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .json(competitions_path)
)

print(f"✅ Competitions loaded: {competitions_df.count()}")
display(competitions_df.limit(5))

📥 Reading: dbfs:/statsbomb-data/competitions.json
✅ Competitions loaded: 75


competition_gender,competition_id,competition_international,competition_name,competition_youth,country_name,match_available,match_available_360,match_updated,match_updated_360,season_id,season_name
male,9,false,1. Bundesliga,false,Germany,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,281,2023/2024
male,9,false,1. Bundesliga,false,Germany,2024-05-19T11:11:14.192381,null,2024-05-19T11:11:14.192381,null,27,2015/2016
male,1267,true,African Cup of Nations,false,Africa,2024-09-28T01:57:35.846538,null,2024-09-28T01:57:35.846538,null,107,2023
male,16,false,Champions League,false,Europe,2025-05-08T15:10:50.835274,null,2025-05-08T15:10:50.835274,2021-06-13T16:17:31.694,4,2018/2019
male,16,false,Champions League,false,Europe,2024-02-13T02:35:28.134882,null,2024-02-13T02:35:28.134882,2021-06-13T16:17:31.694,1,2017/2018


In [0]:
# Clean and save to Snowflake
competitions_clean = competitions_df.select(
    col("competition_id"),
    col("competition_name"),
    col("season_id"),
    col("season_name"),
    col("country_name").alias("country"),
    col("competition_gender").alias("gender")
)

In [0]:
# Write to Snowflake
print("📤 Writing to Snowflake...")
competitions_clean.write \
    .format("snowflake") \
    .options(**SNOWFLAKE_OPTIONS) \
    .option("dbtable", "DIM_COMPETITIONS") \
    .mode("overwrite") \
    .save()

📤 Writing to Snowflake...


## ⚽ 2. Load All Matches

In [0]:
matches_path = f"{DATA_PATH}/matches/*/*.json"
print(f"📥 Reading: {matches_path}")

matches_df = (
    spark.read
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .json(matches_path)
    .cache()
)

print(f"⚽ Total Matches: {matches_df.count():,}")
display(matches_df.limit(5))


📥 Reading: dbfs:/statsbomb-data/matches/*/*.json
⚽ Total Matches: 3,464


away_score,away_team,competition,competition_stage,home_score,home_team,kick_off,last_updated,last_updated_360,match_date,match_id,match_status,match_status_360,match_week,metadata,referee,season,stadium
2,"List(male, null, 322, Eibar, List(214, Spain), List(List(List(214, Spain), 1961-03-14, 221, José Luis Mendilibar Etxebarria, José Luis Mendilibar)))","List(11, La Liga, Spain)","List(1, Regular Season)",2,"List(List(214, Spain), male, null, 221, Levante UD, List(List(List(214, Spain), 1966-06-21, 1612, Luis Lucas Alcaraz González, Lucas Alcaraz)))",20:00:00.000,2023-02-21T15:19:45.745471,null,2015-09-23,3825848,available,unscheduled,5,"List(1.1.0, 2, 2)",null,"List(27, 2015/2016)","List(List(214, Spain), 350, Estadio Ciudad de Valencia)"
0,"List(male, null, 213, Sevilla, List(214, Spain), List(List(List(214, Spain), 1971-11-03, 21, Unai Emery Etxegoien, Unai Emery)))","List(11, La Liga, Spain)","List(1, Regular Season)",2,"List(List(214, Spain), male, null, 208, Las Palmas, List(List(List(214, Spain), 1953-12-02, 745, Francisco Herrera Lorenzo, Paco Herrera)))",22:00:00.000,2023-02-21T15:19:07.927548,null,2015-09-23,3825895,available,unscheduled,5,"List(1.1.0, 2, 2)","List(List(214, Spain), 2728, Carlos del Cerro Grande)","List(27, 2015/2016)","List(List(214, Spain), 357, Estadio de Gran Canaria)"
2,"List(male, null, 216, Getafe, List(214, Spain), List(List(List(11, Argentina), 1973-03-05, 1001036, Juan Eduardo Esnáider Belén, null)))","List(11, La Liga, Spain)","List(1, Regular Season)",0,"List(List(214, Spain), male, null, 219, RC Deportivo La Coruña, List(List(List(214, Spain), 1976-02-23, 3157, Víctor Sánchez del Amo, Víctor Sánchez)))",18:15:00.000,2022-08-30T12:53:16.828350,null,2016-05-01,3825894,available,unscheduled,36,"List(1.1.0, 2, 2)","List(List(214, Spain), 1008, Carlos Clos Gómez)","List(27, 2015/2016)","List(List(214, Spain), 4658, Estadio Abanca-Riazor)"
1,"List(male, null, 221, Levante UD, List(214, Spain), List(List(List(214, Spain), 1970-02-05, 677, Joan Francesc Ferrer Sicilia, Rubí)))","List(11, La Liga, Spain)","List(1, Regular Season)",3,"List(List(214, Spain), male, null, 223, Málaga, List(List(List(214, Spain), 1970-05-01, 188, Javier Gracia Carlos, Javi Gracia)))",20:30:00.000,2022-08-30T12:53:53.790835,null,2016-05-02,3825855,available,unscheduled,36,"List(1.1.0, 2, 2)","List(List(214, Spain), 2480, Iñaki Bikandi Garrido)","List(27, 2015/2016)","List(List(214, Spain), 346, Estadio La Rosaleda)"
2,"List(male, null, 322, Eibar, List(214, Spain), List(List(List(214, Spain), 1961-03-14, 221, José Luis Mendilibar Etxebarria, José Luis Mendilibar)))","List(11, La Liga, Spain)","List(1, Regular Season)",4,"List(List(214, Spain), male, null, 214, Espanyol, List(List(List(187, Romania), 1972-03-08, 1816, Constantin Gâlcă, null)))",19:00:00.000,2022-04-06T22:39:49.425989,null,2016-05-15,3825908,available,unscheduled,38,"List(1.1.0, 2, 2)","List(List(214, Spain), 2692, Mario Melero López)","List(27, 2015/2016)","List(List(214, Spain), 351, RCDE Stadium)"


In [0]:
# Transform matches
matches_clean = matches_df.select(
    col("match_id"),
    col("match_date"),
    col("kick_off"),
    col("match_week"),
    col("match_status"),
    
    # Competition
    col("competition.competition_id").alias("competition_id"),
    col("competition.competition_name").alias("competition_name"),
    
    # Season
    col("season.season_id").alias("season_id"),
    col("season.season_name").alias("season_name"),
    
    # Teams
    col("home_team.home_team_id").alias("home_team_id"),
    col("home_team.home_team_name").alias("home_team_name"),
    col("away_team.away_team_id").alias("away_team_id"),
    col("away_team.away_team_name").alias("away_team_name"),
    
    # Scores
    col("home_score"),
    col("away_score"),
    
    # Stadium
    col("stadium.name").alias("stadium_name"),
    col("stadium.country.name").alias("stadium_country")
).withColumn(
    "winner",
    when(col("home_score") > col("away_score"), "HOME")
    .when(col("away_score") > col("home_score"), "AWAY")
    .otherwise("DRAW")
).withColumn(
    "total_goals",
    col("home_score") + col("away_score")
)

display(matches_clean.limit(10))

match_id,match_date,kick_off,match_week,match_status,competition_id,competition_name,season_id,season_name,home_team_id,home_team_name,away_team_id,away_team_name,home_score,away_score,stadium_name,stadium_country,winner,total_goals
3825848,2015-09-23,20:00:00.000,5,available,11,La Liga,27,2015/2016,221,Levante UD,322,Eibar,2,2,Estadio Ciudad de Valencia,Spain,DRAW,4
3825895,2015-09-23,22:00:00.000,5,available,11,La Liga,27,2015/2016,208,Las Palmas,213,Sevilla,2,0,Estadio de Gran Canaria,Spain,HOME,2
3825894,2016-05-01,18:15:00.000,36,available,11,La Liga,27,2015/2016,219,RC Deportivo La Coruña,216,Getafe,0,2,Estadio Abanca-Riazor,Spain,AWAY,2
3825855,2016-05-02,20:30:00.000,36,available,11,La Liga,27,2015/2016,223,Málaga,221,Levante UD,3,1,Estadio La Rosaleda,Spain,HOME,4
3825908,2016-05-15,19:00:00.000,38,available,11,La Liga,27,2015/2016,214,Espanyol,322,Eibar,4,2,RCDE Stadium,Spain,HOME,6
3825883,2016-05-15,12:00:00.000,38,available,11,La Liga,27,2015/2016,223,Málaga,208,Las Palmas,4,1,Estadio La Rosaleda,Spain,HOME,5
3825900,2016-05-15,19:30:00.000,38,available,11,La Liga,27,2015/2016,1041,Sporting Gijón,222,Villarreal,2,0,Estadio Municipal El Molinón,Spain,HOME,2
3825902,2016-05-15,19:30:00.000,38,available,11,La Liga,27,2015/2016,360,Rayo Vallecano,221,Levante UD,3,1,Estadio de Vallecas,Spain,HOME,4
3825876,2016-05-15,19:30:00.000,38,available,11,La Liga,27,2015/2016,218,Real Betis,216,Getafe,2,1,Estadio Benito Villamarín,Spain,HOME,3
3825846,2016-05-14,17:00:00.000,38,available,11,La Liga,27,2015/2016,219,RC Deportivo La Coruña,220,Real Madrid,0,2,Estadio Abanca-Riazor,Spain,AWAY,2


In [0]:
# Write to Snowflake
print("📤 Writing matches to Snowflake...")
matches_clean.write \
    .format("snowflake") \
    .options(**SNOWFLAKE_OPTIONS) \
    .option("dbtable", "FACT_MATCHES") \
    .mode("overwrite") \
    .save()

print("✅ FACT_MATCHES created in Snowflake!")

📤 Writing matches to Snowflake...
✅ FACT_MATCHES created in Snowflake!


## 🔴 3. Find Manchester United Matches

In [0]:
# Filter Man United
mu_matches = matches_clean.filter(
    (lower(col("home_team_name")).contains("manchester united")) | 
    (lower(col("away_team_name")).contains("manchester united"))
)

mu_count = mu_matches.count()
print(f"🔴 Manchester United Matches: {mu_count}")

if mu_count > 0:
    display(mu_matches)
    
    # Save to Snowflake
    mu_matches.write \
        .format("snowflake") \
        .options(**SNOWFLAKE_OPTIONS) \
        .option("dbtable", "MANUTD_MATCHES") \
        .mode("overwrite") \
        .save()
    
    print("✅ MANUTD_MATCHES saved!")
else:
    print("⚠️  No direct Man United matches in dataset")
    print("   (StatsBomb Open Data focuses on international & women's competitions)")

🔴 Manchester United Matches: 79


match_id,match_date,kick_off,match_week,match_status,competition_id,competition_name,season_id,season_name,home_team_id,home_team_name,away_team_id,away_team_name,home_score,away_score,stadium_name,stadium_country,winner,total_goals
3754014,2015-12-26,13:45:00.000,18,available,2,Premier League,27,2015/2016,30,Stoke City,39,Manchester United,2,0,bet365 Stadium,England,HOME,2
3754347,2015-12-05,16:00:00.000,15,available,2,Premier League,27,2015/2016,39,Manchester United,40,West Ham United,0,0,Old Trafford,England,DRAW,0
3754015,2015-10-25,15:05:00.000,10,available,2,Premier League,27,2015/2016,39,Manchester United,36,Manchester City,0,0,Old Trafford,England,DRAW,0
3754008,2015-09-26,16:00:00.000,7,available,2,Premier League,27,2015/2016,39,Manchester United,41,Sunderland,3,0,Old Trafford,England,HOME,3
3754011,2015-09-20,17:00:00.000,6,available,2,Premier League,27,2015/2016,25,Southampton,39,Manchester United,2,3,St. Mary''s Stadium,England,AWAY,5
3754107,2015-08-14,20:45:00.000,2,available,2,Premier League,27,2015/2016,59,Aston Villa,39,Manchester United,0,1,Villa Park,England,AWAY,1
3754097,2015-08-08,13:45:00.000,1,available,2,Premier League,27,2015/2016,39,Manchester United,38,Tottenham Hotspur,1,0,Old Trafford,England,HOME,1
3754185,2015-11-21,13:45:00.000,13,available,2,Premier League,27,2015/2016,23,Watford,39,Manchester United,1,2,Vicarage Road,England,AWAY,3
3754162,2015-12-12,18:30:00.000,16,available,2,Premier League,27,2015/2016,28,AFC Bournemouth,39,Manchester United,2,1,Vitality Stadium,England,HOME,3
3754171,2015-10-04,17:00:00.000,8,available,2,Premier League,27,2015/2016,1,Arsenal,39,Manchester United,3,0,Emirates Stadium,England,HOME,3


✅ MANUTD_MATCHES saved!


## 🎯 4. Process Events (Sample)

In [0]:
# Get event files with correct path
events_path = f"{DATA_PATH}/events/"
event_files = dbutils.fs.ls(events_path)
total_files = len(event_files)

print(f"📂 Total Event Files: {total_files:,}")

# Process first 100 matches (manageable size)
sample_size = __builtins__.min(100, total_files)
sample_paths = [f.path for f in event_files[:sample_size]]

print(f"📥 Loading {sample_size} matches...")

# Read events with proper options
events_df = (
    spark.read
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .json(sample_paths)
    .cache()
)

total_events = events_df.count()
print(f"🎯 Total Events: {total_events:,}")



📂 Total Event Files: 3,464
📥 Loading 100 matches...
🎯 Total Events: 353,140


In [0]:
from pyspark.sql.functions import input_file_name, regexp_extract, col

sample_paths_file = [p.replace("dbfs:/", "file:/dbfs/") for p in sample_paths]

events_df = (spark.read
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .json(sample_paths_file)
)

events_with_match = events_df.withColumn(
    "_fn", input_file_name()
).withColumn(
    "match_id",
    regexp_extract(col("_fn"), r'(\d+)\.json$', 1).cast("long")
)

events_with_match.select("_fn", "match_id").show(10, False)


+-------------------------------------------+--------+
|_fn                                        |match_id|
+-------------------------------------------+--------+
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
|file:/dbfs/statsbomb-data/events/18243.json|18243   |
+-------------------------------------------+--------+
only showing top 10 rows



In [0]:
# Transform events
events_clean = events_with_match.select(
    col("id").alias("event_id"),
    col("match_id"),
    col("index").alias("event_index"),
    col("period"),
    col("timestamp"),
    col("minute"),
    col("second"),
    col("type.id").alias("event_type_id"),
    col("type.name").alias("event_type"),
    col("possession"),
    col("possession_team.id").alias("possession_team_id"),
    col("possession_team.name").alias("possession_team_name"),
    col("play_pattern.name").alias("play_pattern"),
    col("team.id").alias("team_id"),
    col("team.name").alias("team_name"),
    col("player.id").alias("player_id"),
    col("player.name").alias("player_name"),
    col("position.name").alias("position"),
    col("location")[0].alias("location_x"),
    col("location")[1].alias("location_y"),
    col("duration"),
    col("under_pressure").cast("boolean"),
    col("off_camera").cast("boolean")
)

print(f"✅ Transformed {events_clean.count():,} events")
display(events_clean.limit(10))

✅ Transformed 353,140 events


event_id,match_id,event_index,period,timestamp,minute,second,event_type_id,event_type,possession,possession_team_id,possession_team_name,play_pattern,team_id,team_name,player_id,player_name,position,location_x,location_y,duration,under_pressure,off_camera
aa83c2da-936f-418e-9a9c-76d2b6531412,18243,1,1,00:00:00.000,0,0,35,Starting XI,1,220,Real Madrid,Regular Play,220,Real Madrid,null,null,null,null,null,0.0,null,null
2ce513d7-b156-4951-b054-42b14cc0717d,18243,2,1,00:00:00.000,0,0,35,Starting XI,1,220,Real Madrid,Regular Play,212,Atlético Madrid,null,null,null,null,null,0.0,null,null
5fea79a2-4748-4ac0-b73a-a5f0dddc562b,18243,3,1,00:00:00.000,0,0,18,Half Start,1,220,Real Madrid,Regular Play,220,Real Madrid,null,null,null,null,null,0.0,null,null
5b573194-f57a-438f-9b1d-2b26a832516b,18243,4,1,00:00:00.000,0,0,18,Half Start,1,220,Real Madrid,Regular Play,212,Atlético Madrid,null,null,null,null,null,0.0,null,null
ceae30fc-3f31-4660-aff6-5c2e8adda35c,18243,5,1,00:00:01.450,0,1,30,Pass,2,212,Atlético Madrid,From Kick Off,212,Atlético Madrid,5487,Antoine Griezmann,Left Center Forward,61.0,41.0,0.401292,null,null
39a40855-2a60-4b06-8cbf-d73f4103d238,18243,6,1,00:00:01.851,0,1,42,Ball Receipt*,2,212,Atlético Madrid,From Kick Off,212,Atlético Madrid,5199,Jorge Resurrección Merodio,Left Midfield,61.0,43.0,null,null,null
e15a39cb-eaf8-42bc-9a2a-1389eb25548b,18243,7,1,00:00:01.851,0,1,43,Carry,2,212,Atlético Madrid,From Kick Off,212,Atlético Madrid,5199,Jorge Resurrección Merodio,Left Midfield,61.0,43.0,0.501108,null,null
9de50022-a44d-4212-86a0-13f7fde8801d,18243,8,1,00:00:02.352,0,2,30,Pass,2,212,Atlético Madrid,From Kick Off,212,Atlético Madrid,5199,Jorge Resurrección Merodio,Left Midfield,61.0,44.0,1.6133,null,null
f4f5c37b-bfe1-42e8-8d8b-688b8d94d12a,18243,9,1,00:00:03.309,0,3,17,Pressure,2,212,Atlético Madrid,From Kick Off,220,Real Madrid,19677,Karim Benzema,Center Forward,72.0,38.0,3.468631,null,null
7f81d221-9986-4087-9e8e-3281df452f21,18243,10,1,00:00:03.965,0,3,42,Ball Receipt*,2,212,Atlético Madrid,From Kick Off,212,Atlético Madrid,6384,Gabriel Fernández Arenas,Right Defensive Midfield,45.0,43.0,null,true,null


In [0]:
# Event type distribution
event_types = events_clean.groupBy("event_type") \
    .count() \
    .orderBy(desc("count"))

print("📊 Top Event Types:")
display(event_types.limit(20))


📊 Top Event Types:


event_type,count
Pass,99516
Ball Receipt*,89591
Carry,82199
Pressure,30311
Ball Recovery,11173
Duel,5832
Clearance,3855
Block,3741
Dribble,3603
Goal Keeper,3213


In [0]:
# Write sample to Snowflake
print("📤 Writing events sample to Snowflake...")
events_clean.write \
    .format("snowflake") \
    .options(**SNOWFLAKE_OPTIONS) \
    .option("dbtable", "FACT_EVENTS") \
    .mode("overwrite") \
    .save()

print("✅ FACT_EVENTS created in Snowflake!")

📤 Writing events sample to Snowflake...
✅ FACT_EVENTS created in Snowflake!


## ⚽ 5. Extract Passes


In [0]:
# Filter passes (use events_with_match instead of events_df)
passes = events_with_match.filter(col("type.name") == "Pass").select(
    col("id").alias("event_id"),
    col("match_id"),  # ← Now exists!
    col("player.name").alias("player_name"),
    col("team.name").alias("team_name"),
    col("minute"),
    col("location")[0].alias("pass_start_x"),
    col("location")[1].alias("pass_start_y"),
    col("pass.end_location")[0].alias("pass_end_x"),
    col("pass.end_location")[1].alias("pass_end_y"),
    col("pass.length").alias("pass_length"),
    col("pass.angle").alias("pass_angle"),
    col("pass.height.name").alias("pass_height"),
    col("pass.body_part.name").alias("pass_body_part"),
    col("pass.outcome.name").alias("pass_outcome"),
    col("pass.recipient.name").alias("pass_recipient"),
    coalesce(col("under_pressure"), lit(False)).alias("under_pressure")
)

pass_count = passes.count()
print(f"⚽ Total Passes: {pass_count:,}")

⚽ Total Passes: 99,516


In [0]:
# Pass completion by team
pass_stats = passes.groupBy("team_name") \
    .agg(
        count("*").alias("total_passes"),
        sum(when(col("pass_outcome").isNull(), 1).otherwise(0)).alias("completed"),
        sum(when(col("pass_outcome").isNotNull(), 1).otherwise(0)).alias("failed")
    ) \
    .withColumn(
        "completion_rate",
        round((col("completed") / col("total_passes")) * 100, 2)
    ) \
    .orderBy(desc("total_passes"))

display(pass_stats.limit(20))

team_name,total_passes,completed,failed,completion_rate
Barcelona,25472,22350,3122,87.74
Manchester City WFC,6656,5258,1398,79.0
Arsenal WFC,6504,5267,1237,80.98
Chelsea FCW,5720,4413,1307,77.15
Birmingham City WFC,5478,4269,1209,77.93
Reading WFC,5033,3614,1419,71.81
West Ham United LFC,4407,3029,1378,68.73
Everton LFC,4179,2781,1398,66.55
Brighton & Hove Albion WFC,4064,2729,1335,67.15
Liverpool WFC,3571,2293,1278,64.21


In [0]:
# Save to Snowflake
passes.write \
    .format("snowflake") \
    .options(**SNOWFLAKE_OPTIONS) \
    .option("dbtable", "FACT_PASSES") \
    .mode("overwrite") \
    .save()

print("✅ FACT_PASSES saved to Snowflake!")


✅ FACT_PASSES saved to Snowflake!


## 🎯 6. Extract Shots & xG

In [0]:
# Filter shots (use events_with_match)
shots = events_with_match.filter(col("type.name") == "Shot").select(
    col("id").alias("event_id"),
    col("match_id"),  # ← Now exists!
    col("player.name").alias("player_name"),
    col("team.name").alias("team_name"),
    col("minute"),
    col("location")[0].alias("shot_x"),
    col("location")[1].alias("shot_y"),
    col("shot.statsbomb_xg").alias("xg"),
    col("shot.outcome.name").alias("shot_outcome"),
    col("shot.type.name").alias("shot_type"),
    col("shot.body_part.name").alias("shot_body_part"),
    col("shot.technique.name").alias("shot_technique"),
    coalesce(col("under_pressure"), lit(False)).alias("under_pressure")
)

shot_count = shots.count()
print(f"🎯 Total Shots: {shot_count:,}")


🎯 Total Shots: 2,785


In [0]:

# xG by team
team_xg = shots.groupBy("team_name") \
    .agg(
        count("*").alias("total_shots"),
        sum("xg").alias("total_xg"),
        round(avg("xg"), 3).alias("avg_xg_per_shot"),
        sum(when(col("shot_outcome") == "Goal", 1).otherwise(0)).alias("goals")
    ) \
    .withColumn(
        "conversion_rate",
        round((col("goals") / col("total_shots")) * 100, 2)
    ) \
    .orderBy(desc("total_xg"))

print("📊 Expected Goals (xG) by Team:")
display(team_xg.limit(20))

📊 Expected Goals (xG) by Team:


team_name,total_shots,total_xg,avg_xg_per_shot,goals,conversion_rate
Barcelona,558,66.3955568403,0.119,86,15.41
Arsenal WFC,226,32.2650878936,0.143,45,19.91
Manchester City WFC,232,25.968459956399997,0.112,32,13.79
Chelsea FCW,221,20.9568969887,0.095,12,5.43
Reading WFC,164,15.093922083500006,0.092,19,11.59
West Ham United LFC,114,13.800577324899997,0.121,11,9.65
Real Madrid,101,13.444994187700003,0.133,17,16.83
Everton LFC,137,12.6139825492,0.092,6,4.38
Birmingham City WFC,128,12.5138330415,0.098,14,10.94
Brighton & Hove Albion WFC,105,11.599070251700002,0.11,7,6.67


In [0]:

# Save to Snowflake
shots.write \
    .format("snowflake") \
    .options(**SNOWFLAKE_OPTIONS) \
    .option("dbtable", "FACT_SHOTS") \
    .mode("overwrite") \
    .save()

print("✅ FACT_SHOTS saved to Snowflake!")


✅ FACT_SHOTS saved to Snowflake!


## 📊 7. Final Summary

In [0]:
print("="*60)
print("✅ DATA PIPELINE COMPLETE!")
print("="*60)
print()
print("📊 Data Processed:")
print(f"   Competitions: {competitions_df.count()}")
print(f"   Matches: {matches_clean.count():,}")
print(f"   Events (sample): {total_events:,}")
print(f"   Passes: {pass_count:,}")
print(f"   Shots: {shot_count:,}")
print()
print("❄️  Snowflake Tables Created:")
print("   • DIM_COMPETITIONS")
print("   • FACT_MATCHES")
print("   • FACT_EVENTS")
print("   • FACT_PASSES")
print("   • FACT_SHOTS")
if mu_count > 0:
    print("   • MANUTD_MATCHES")
print()
print("🚀 Ready for Analytics & Visualization!")
print("="*60)

✅ DATA PIPELINE COMPLETE!

📊 Data Processed:


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8898476091502896>, line 6
      4 print()
      5 print("📊 Data Processed:")
----> 6 print(f"   Competitions: {competitions_df.count()}")
      7 print(f"   Matches: {matches_clean.count():,}")
      8 print(f"   Events (sample): {total_events:,}")

NameError: name 'competitions_df' is not defined